# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [3]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [4]:
feature_columns = contract["features"]

# Preprocesador para árboles: solo imputación por mediana (no necesitan escalado)
tree_preprocessor = ColumnTransformer(
    [("numeric", Pipeline([("imputer", SimpleImputer(strategy="median"))]), feature_columns)],
    verbose_feature_names_out=False,
)

# Preprocesador para modelos sensibles a la escala: imputación + StandardScaler
scaled_preprocessor = ColumnTransformer(
    [("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), feature_columns)],
    verbose_feature_names_out=False,
)

**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

> Si se ajusta el `StandardScaler` con todo `X_train` antes de la validación cruzada, la media y la desviación estándar se calculan usando también las filas que luego quedarán en el fold de validación. Esa información "se filtra" hacia el entrenamiento (*data leakage*) y las métricas de CV salen un poco optimistas, porque el fold de validación ya no es realmente datos nuevos. Dentro del `Pipeline`, `cross_validate` y `GridSearchCV` ajustan el escalador **solo con las filas de entrenamiento de cada fold** y luego aplican esa transformación al fold de validación, que es lo que ocurriría con datos reales en producción. Además, al guardar el pipeline completo, el preprocesamiento viaja con el modelo y la inferencia aplica exactamente la misma transformación.

## 3. Modelos candidatos

In [5]:
models = {
    "decision_tree": Pipeline([
        ("preprocess", clone(tree_preprocessor)),
        ("classifier", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]),
    "logistic_regression": Pipeline([
        ("preprocess", clone(scaled_preprocessor)),
        ("classifier", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ]),
    "knn": Pipeline([
        ("preprocess", clone(scaled_preprocessor)),
        ("classifier", KNeighborsClassifier()),
    ]),
}
list(models)

['decision_tree', 'logistic_regression', 'knn']

## 4. Comparación inicial con validación cruzada

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rows = []
for name, pipe in models.items():
    scores = cross_validate(
        pipe, X_train, y_train, cv=cv, scoring="accuracy",
        return_train_score=True, n_jobs=-1,
    )
    rows.append({
        "model": name,
        "train_accuracy": scores["train_score"].mean(),
        "val_accuracy": scores["test_score"].mean(),
        "val_std": scores["test_score"].std(),
    })

initial_comparison = (
    pd.DataFrame(rows)
    .sort_values("val_accuracy", ascending=False)
    .reset_index(drop=True)
)
initial_comparison["gap_train_val"] = initial_comparison["train_accuracy"] - initial_comparison["val_accuracy"]
initial_comparison.round(4)

,model,train_accuracy,val_accuracy,val_std,gap_train_val
0,logistic_regression,1.0000,0.9791,0.0277,0.0209
1,knn,0.9771,0.9507,0.0353,0.0264
2,decision_tree,1.0000,0.9022,0.0586,0.0978


## 5. Optimización de hiperparámetros

In [7]:
param_grids = {
    "decision_tree": {
        "classifier__max_depth": [2, 3, 4, 5, None],
        "classifier__min_samples_leaf": [1, 2, 5],
    },
    "logistic_regression": {
        "classifier__C": [0.01, 0.1, 1, 10, 100],
    },
    "knn": {
        "classifier__n_neighbors": [3, 5, 7, 9, 11],
        "classifier__weights": ["uniform", "distance"],
    },
}

searches = {}
rows = []
for name, pipe in models.items():
    search = GridSearchCV(
        pipe, param_grids[name], cv=cv, scoring="accuracy",
        n_jobs=-1, return_train_score=True,
    )
    search.fit(X_train, y_train)
    searches[name] = search

    best = search.best_index_
    rows.append({
        "model": name,
        "cv_accuracy": search.best_score_,
        "cv_std": search.cv_results_["std_test_score"][best],
        "train_accuracy": search.cv_results_["mean_train_score"][best],
        "best_params": search.best_params_,
    })

tuned_comparison = (
    pd.DataFrame(rows)
    .sort_values(["cv_accuracy", "cv_std"], ascending=[False, True])
    .reset_index(drop=True)
)
tuned_comparison["gap_train_val"] = tuned_comparison["train_accuracy"] - tuned_comparison["cv_accuracy"]
tuned_comparison.round(4)

,model,cv_accuracy,cv_std,train_accuracy,best_params,gap_train_val
0,logistic_regression,0.9791,0.0277,1.0000,{'classifier__C': 1},0.0209
1,knn,0.9579,0.0405,0.9648,"{'classifier__n_neighbors': 7, 'classifier__we...",0.0069
2,decision_tree,0.9025,0.0762,0.9859,"{'classifier__max_depth': 5, 'classifier__min_...",0.0835


## 6. Selección y persistencia del campeón

In [8]:
# Campeón: mayor best_score_ de CV (desempata la menor desviación estándar)
champion_name = tuned_comparison.loc[0, "model"]
champion = searches[champion_name].best_estimator_

joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

training_metadata = {
    "champion": champion_name,
    "selection_metric": "accuracy",
    "cv_folds": 5,
    "cv_score": float(searches[champion_name].best_score_),
    "best_params": searches[champion_name].best_params_,
    "random_state": RANDOM_STATE,
}
(ARTIFACTS_DIR / "training_metadata.json").write_text(
    json.dumps(training_metadata, indent=2, default=str), encoding="utf-8"
)

tuned_comparison.to_csv(REPORTS_DIR / "cv_results.csv", index=False)

print("Modelo campeón:", champion_name)
print("Accuracy CV:", round(training_metadata["cv_score"], 4))
print("Mejores hiperparámetros:", training_metadata["best_params"])

Modelo campeón: logistic_regression
Accuracy CV: 0.9791
Mejores hiperparámetros: {'classifier__C': 1}


In [9]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

> Se eligió **regresión logística** (`C=1`, con `StandardScaler` dentro del pipeline).
>
> - **Desempeño:** obtuvo el mayor accuracy de validación cruzada (≈ 0.979), por encima de KNN (≈ 0.958) y del árbol de decisión (≈ 0.903). Ya era la mejor antes de optimizar hiperparámetros, así que la elección no depende de la búsqueda.
> - **Variabilidad:** también tiene la menor desviación estándar entre folds (≈ 0.028, frente a ≈ 0.041 de KNN y ≈ 0.076 del árbol), es decir, es la más estable ante distintas particiones.
> - **Sobreajuste:** la brecha entre train y validación es pequeña (≈ 0.02). El árbol es el que más se sobreajusta (≈ 0.08 de brecha) y su rendimiento depende mucho del fold. KNN casi no tiene brecha, pero rinde menos.
> - **Complejidad:** el modelo es lineal con un solo hiperparámetro relevante, así que es barato de entrenar y de servir. KNN debe guardar todo el conjunto de entrenamiento y calcular distancias en cada predicción.
> - **Interpretabilidad:** los coeficientes dan el peso de cada variable por clase. Hay que leerlos con cuidado porque varias variables están correlacionadas (p. ej. `total_phenols` y `flavanoids`), pero siguen siendo más fáciles de explicar que una decisión de KNN.
>
> Nota: con solo 142 muestras de entrenamiento, la diferencia entre logística y KNN es de pocas muestras, por lo que la ventaja no es enorme; se decide por desempeño, estabilidad y simplicidad en conjunto. La elección se hizo solo con validación cruzada, sin tocar `test.csv`.

## Checklist

- [x] Todo el procesamiento está dentro de pipelines.
- [x] Comparé tres familias de modelos.
- [x] Optimicé hiperparámetros con CV.
- [x] Elegí el campeón sin usar test.
- [x] Guardé el pipeline completo y sus metadatos.